**Objective**

Extract standardized image quality and geometric features from each dataset independently to create model-ready feature datasets.

Input

data/samples/
    ├── fairface/
    ├── celeba/
    ├── driver_drowsiness/
    └── driver_distraction/

Output

data/processed/

feature_fairface.csv
feature_celeba.csv
feature_driver_drowsiness.csv
feature_driver_distraction.csv

In [ ]:
import cv2
print(cv2.__version__)

In [ ]:
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from tqdm import tqdm

In [ ]:
PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

SAMPLE_DIR = PROJECT_DIR / "data" / "samples"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
DATASETS = {

    "fairface":
        SAMPLE_DIR / "fairface",

    "celeba":
        SAMPLE_DIR / "celeba",

    "driver_drowsiness":
        SAMPLE_DIR / "driver_drowsiness",

    "driver_distraction":
        SAMPLE_DIR / "driver_distraction"

}

In [ ]:
def extract_features(image_path):

    image = cv2.imread(str(image_path))

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    height, width = gray.shape

    brightness = np.mean(gray)

    contrast = np.std(gray)

    blur_score = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    size_mb = image_path.stat().st_size / (1024 * 1024)

    image_area = width * height

    return {

        "file_name": image_path.name,

        "size_mb": size_mb,

        "width": width,

        "height": height,

        "aspect_ratio": width / height,

        "brightness": brightness,

        "contrast": contrast,

        "blur_score": blur_score,

        "image_area": image_area,

        "size_per_pixel": size_mb / image_area,

        "is_square": int(abs(width-height) < 10)

    }

In [ ]:
def process_dataset(dataset_folder):

    image_dir = dataset_folder / "images"

    features = []

    for image in tqdm(sorted(image_dir.glob("*"))):

        try:

            features.append(
                extract_features(image)
            )

        except Exception:

            continue

    return pd.DataFrame(features)

______________________________Processing all the dataset using above defined functions____________________________

In [ ]:
feature_tables = {}

for name, folder in DATASETS.items():

    print(f"\nProcessing {name}")

    feature_tables[name] = process_dataset(folder)

In [ ]:
for name in feature_tables:

    metadata = pd.read_csv(
        DATASETS[name] / "metadata.csv"
    )

    feature_tables[name] = feature_tables[name].merge(

        metadata,

        on="file_name",

        how="left"

    )

In [ ]:
for name, df in feature_tables.items():

    output = PROCESSED_DIR / f"feature_{name}.csv"

    df.to_csv(
        output,
        index=False
    )

    print(output.name, df.shape)

In [ ]:
for name in feature_tables:

    print("="*60)

    print(name)

    print(feature_tables[name].head())